# DSPy ExtractAgent — one ReAct round, prompts, and tool text

This notebook matches **your** code paths (not generic advice).

## Where `ExtractAgent` is created and run

1. **`LerimRuntime._sync_inner`** (`src/lerim/server/runtime.py`) builds `RuntimeContext`, then `ExtractAgent(ctx)`, then calls **`_run_with_fallback(flow="sync", module=agent, input_args={...})`**.
2. **`_run_with_fallback`** wraps the call in **`with dspy.context(lm=lm, adapter=dspy.XMLAdapter()): return module(**input_args)`** — same adapter as `extract.py`’s `__main__` smoke test.
3. **`ExtractAgent`** (`src/lerim/agents/extract.py`) sets **`self.react = dspy.ReAct(ExtractSignature, tools=make_extract_tools(ctx), max_iters=ctx.config.lead_role.max_iters_sync)`** and **`forward`** forwards keyword args to **`self.react(...)`**.
4. **Daemon** (`src/lerim/server/daemon.py`): **`LerimRuntime(default_cwd=repo_path)`** then **`agent.sync(Path(session_path), memory_root=project_memory)`** — production sync entry.

## After a real sync run

The runtime writes **`run_folder / "agent_trace.json"`** from **`prediction.trajectory`** (`runtime.py`). Open that file to see thought / tool / observation steps without this notebook.

## DSPy: seeing prompts (official)

- **`dspy.inspect_history(n)`** prints the last *n* **LLM** calls from DSPy’s global history ([API](https://dspy.ai/api/utils/inspect_history/), [observability tutorial](https://dspy.ai/tutorials/observability/)).  
- The tutorial notes **`inspect_history` only logs LLM calls** — not separate spans for tools/retrievers; for richer traces use MLflow **`mlflow.dspy.autolog()`** or DSPy **`dspy.configure(callbacks=[...])`** (`BaseCallback`: `on_lm_*`, `on_tool_*`, `on_adapter_*`).

**Requirement:** configured **`[roles.lead]`** and API keys as for normal Lerim (e.g. `~/.lerim/config.toml` + env). **`max_iters_sync=1`** below limits ReAct iterations to **one** (one loop round of the agent).

In [1]:
from __future__ import annotations

import json
import tempfile
from dataclasses import replace
from pathlib import Path

import dspy

from lerim.agents.context import RuntimeContext
from lerim.agents.extract import ExtractAgent
from lerim.agents.tools import make_extract_tools
from lerim.config.providers import build_dspy_lm
from lerim.config.settings import get_config

In [2]:
# Same layout as lerim/agents/extract.py __main__, but cap ReAct to one iteration.
cfg = get_config()
cfg = replace(cfg, lead_role=replace(cfg.lead_role, max_iters_sync=5))

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp).resolve()
    mem = (root / "memories").resolve()
    mem.mkdir()
    ws = (root / "workspace").resolve()
    trace_dir = (root / "traces").resolve()
    trace_dir.mkdir()
    run_folder = (ws / "notebook-run").resolve()
    run_folder.mkdir(parents=True)

    trace_path = (trace_dir / "session.jsonl").resolve()
    turns = [
        ("user", "Remember: we use uv, never raw pip, for installs in docs."),
        ("assistant", "I'll document uv sync / uv add in README."),
    ]
    trace_path.write_text(
        "\n".join(json.dumps({"role": r, "content": t}) for r, t in turns) + "\n",
        encoding="utf-8",
    )

    artifact_paths = {
        "summary": run_folder / "summary.json",
        "memory_actions": run_folder / "memory_actions.json",
        "agent_log": run_folder / "agent.log",
        "subagents_log": run_folder / "subagents.log",
        "session_log": run_folder / "session.log",
    }
    artifact_paths["summary"].write_text(
        json.dumps({"summary_path": ""}), encoding="utf-8"
    )

    ctx = RuntimeContext(
        config=cfg,
        repo_root=root,
        memory_root=mem,
        workspace_root=ws,
        run_folder=run_folder,
        extra_read_roots=(trace_dir,),
        run_id="notebook-run",
        trace_path=trace_path,
        artifact_paths=artifact_paths,
    )
    memory_index_path = mem / "MEMORY.md"

    agent = ExtractAgent(ctx)
    lm = build_dspy_lm("lead", config=cfg)

    with dspy.context(lm=lm, adapter=dspy.XMLAdapter()):
        pred = agent(
            trace_path=str(trace_path),
            memory_root=str(mem),
            run_folder=str(run_folder),
            memory_actions_path=str(artifact_paths["memory_actions"]),
            memory_index_path=str(memory_index_path),
            run_id="notebook-run",
        )

print("completion_summary:", getattr(pred, "completion_summary", None))
traj = getattr(pred, "trajectory", None) or {}
print("trajectory keys (sample):", sorted(traj.keys())[:20], "... total", len(traj))

completion_summary: Created feedback memory documenting that uv (uv sync/uv add) should be used instead of raw pip for documentation installations. Updated memory index and wrote session summary.
trajectory keys (sample): ['observation_0', 'observation_1', 'observation_2', 'observation_3', 'observation_4', 'thought_0', 'thought_1', 'thought_2', 'thought_3', 'thought_4', 'tool_args_0', 'tool_args_1', 'tool_args_2', 'tool_args_3', 'tool_args_4', 'tool_name_0', 'tool_name_1', 'tool_name_2', 'tool_name_3', 'tool_name_4'] ... total 20


In [3]:
# Full trajectory dict (ReAct: thought_i, tool_name_i, observation_i, ...)
import pprint

pprint.pp(dict(traj))

{'thought_0': 'I need to first scan the existing memory manifest to understand '
              'what memories already exist, then read the session trace to '
              "identify what's worth remembering for future sessions.",
 'tool_name_0': 'scan_memory_manifest',
 'tool_args_0': {},
 'observation_0': '{\n  "count": 0,\n  "memories": []\n}',
 'thought_1': 'The memory manifest is empty. Now I need to read the session '
              "trace to identify what's worth remembering. Let me read the "
              'trace file to understand what happened in this session.',
 'tool_name_1': 'read_file',
 'tool_args_1': {'file_path': '/private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp75oghsxp/traces/session.jsonl'},
 'observation_1': '{"role": "user", "content": "Remember: we use uv, never raw '
                  'pip, for installs in docs."}\n'
                  '{"role": "assistant", "content": "I\'ll document uv sync / '
                  'uv add in README."}\n',
 'thought_2': "

In [6]:
# LLM prompts / responses: last N entries in DSPy global history (see dspy.ai docs)
dspy.inspect_history(n=5)





[2026-04-03T08:23:07.998554]

System message:

Your input fields are:
1. `trace_path` (str): Absolute path to the session trace file
2. `memory_root` (str): Absolute path to the memory root directory
3. `run_folder` (str): Absolute path to the run workspace folder
4. `memory_actions_path` (str): Path for memory_actions.json workspace artifact (optional; runtime may default)
5. `memory_index_path` (str): Path to MEMORY.md index file
6. `run_id` (str): Unique run identifier
7. `trajectory` (str):
Your output fields are:
1. `next_thought` (str): 
2. `next_tool_name` (Literal['read_file', 'read_trace', 'grep_trace', 'scan_memory_manifest', 'write_memory', 'edit_memory', 'update_memory_index', 'write_summary', 'list_files', 'finish']): 
3. `next_tool_args` (dict[str, Any]):
All interactions will be structured in the following way, with the appropriate values filled in.

<trace_path>
{trace_path}
</trace_path>

<memory_root>
{memory_root}
</memory_root>

<run_folder>
{run_folder}
</run_f

In [5]:
# How DSPy sees one bound tool: name + docstring (your _bind copies __name__ / __doc__)
tools = make_extract_tools(ctx)
read_tool = dspy.Tool(tools[0])
print("tool[0] __name__ from list:", getattr(tools[0], "__name__", None))
print("dspy.Tool.name:", read_tool.name)
print("dspy.Tool.desc (first 500 chars):\n", (read_tool.desc or "")[:500])

tool[0] __name__ from list: read_file
dspy.Tool.name: read_file
dspy.Tool.desc (first 500 chars):
 Read a memory file's full text content.

    For large session traces, use read_trace() or grep_trace() instead.

    Args:
        file_path: Absolute path to the file.
    
